### Install Libraries

In [ ]:
# !!pip install torch transformers datasets peft trl accelerate bitsandbytes

### Import Libraries

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOTrainer, DPOConfig

In [ ]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

### Load Model and Tokenizer

In [ ]:
# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto")
ref_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto")  # frozen reference
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

### Load Dataset

In [ ]:
# Dataset must have "prompt", "chosen", "rejected" columns
# dataset = load_dataset("trl-internal-testing/hh-rlhf-helpful-base-trl-style", split="train")
dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="train")

### Setting DPO config and trainer 

In [ ]:
training_args = DPOConfig(
    output_dir="./dpo-full",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-7,        # Lower LR than SFT — DPO is sensitive
    beta=0.1,                  # KL penalty coefficient
    max_length=1024,
    # max_prompt_length=512,
    bf16=True,
    logging_steps=10,
    save_steps=200,
)

In [ ]:
trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,       # Frozen reference model
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)


### to start training run command below

In [ ]:
trainer.train()

### Saved trained model

In [ ]:
# ============================================================
# SAVE
# ============================================================
trainer.save_model("./dpo-full-saved")
tokenizer.save_pretrained("./dpo-full-saved")

### Below Step is to Load trained model and predict based on query

In [ ]:
# ============================================================
# LOAD
# ============================================================
from transformers import AutoModelForCausalLM, AutoTokenizer

# Only load the trained policy model — ref_model not needed at inference
model = AutoModelForCausalLM.from_pretrained(
    "./dpo-full-saved",
    torch_dtype="auto",
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained("./dpo-full-saved")

In [ ]:
# ============================================================
# PREDICT
# ============================================================
import torch

def predict(prompt: str, max_new_tokens: int = 200) -> str:
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(predict("How do I stay motivated when learning something difficult?"))